# Practical P17: Rate Limit Handling & Retry Logic
**Learning Outcome**: Build production-grade retry logic for rate-limited AI APIs.

## Part 1: Starting the Mock LLM server with Rate Limit simulation
We will configure a local server that yields `429 Too Many Requests` status codes for the first two attempts, and succeeds on the third attempt.


In [ ]:
import http.server
import socketserver
import threading
import json
import time

attempt_counts = {}

class RateLimitMockHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass
        
    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length)
        payload = json.loads(body.decode('utf-8'))
        prompt = payload.get('messages', [{}])[-1].get('content', '')
        
        # Keep track of retry attempts for this prompt to simulate temporary rate-limiting
        attempt = attempt_counts.get(prompt, 0) + 1
        attempt_counts[prompt] = attempt
        
        if attempt < 3:
            self.send_response(429)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(b'{"error": "Rate limit exceeded. Try again in 2 seconds."}')
            return
            
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        resp = {
            'choices': [{'message': {'content': f'Response after {attempt} attempts.'}}],
            'usage': {'total_tokens': 10}
        }
        self.wfile.write(json.dumps(resp).encode('utf-8'))

PORT = 8089
server = socketserver.TCPServer(('127.0.0.1', PORT), RateLimitMockHandler)
server.allow_reuse_address = True
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print('Rate limit mock server running on http://127.0.0.1:8089')


## Part 2: Implementing Exponential Backoff
Exponential backoff increases the delay exponentially between consecutive retries: `delay = initial_wait * (2 ** attempt)`.
Let's write a decorator to implement this pattern automatically.


In [ ]:
import logging
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('RetryLogger')

def retry_with_backoff(max_retries=5, initial_wait=1):
    def decorator(func):
        def wrapper(*args, **kwargs):
            wait = initial_wait
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        logger.error('Max retries exceeded!')
                        raise
                    logger.warning(f'Attempt {attempt+1} failed with error: {e}. Retrying in {wait} seconds...')
                    time.sleep(wait)
                    wait *= 2
        return wrapper
    return decorator


## Hands-On Exercise
**Task**: Write a function `query_with_retry(prompt)` that sends a POST to `http://127.0.0.1:8089`. It must use `.raise_for_status()` so HTTP errors like 429 are raised as exceptions.
Apply the `@retry_with_backoff` decorator to this function and execute it. Watch the console to see the warning backoff logging in action.


In [ ]:
# TODO: Apply decorator and run function
import requests

@retry_with_backoff(max_retries=4, initial_wait=1)
def query_with_retry(prompt):
    url = 'http://127.0.0.1:8089'
    payload = {'messages': [{'content': prompt}]}
    r = requests.post(url, json=payload)
    r.raise_for_status()
    return r.json()

try:
    ans = query_with_retry('Simulated Rate Limit API test prompt')
    print('Final API Output:', ans['choices'][0]['message']['content'])
except Exception as e:
    print('Failed:', e)


In [ ]:
server.shutdown()
server.server_close()
print('Rate limit mock server stopped.')
